In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install ultralytics  # Ensure YOLO is installed

from ultralytics import YOLO
import os
import shutil


In [ ]:
!pip install pydrive google-auth google-auth-oauthlib google-auth-httplib2
!pip install pyTelegramBotAPI PyDrive2
!pip install -U PyDrive



In [ ]:
import os

# 🔹 Define Image Folder in Drive
image_folder = "/content/drive/My Drive/Telegram_Images"
os.makedirs(image_folder, exist_ok=True)

print(f"✅ Save images in: {image_folder}")


✅ Save images in: /content/drive/My Drive/Telegram_Images


In [ ]:
!pip install pydrive2




In [ ]:
!pip install googletrans==4.0.0-rc1



In [58]:
import telebot
import requests
import threading
import os
import requests
from telebot.types import InlineKeyboardMarkup, InlineKeyboardButton
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from oauth2client.service_account import ServiceAccountCredentials
import cv2
import torch
import numpy as np
import json
from ultralytics import YOLO
from telebot import types
from googletrans import Translator




# Bot Token & Google Drive Folder ID
BOT_TOKEN = "8187455463:AAG-XxnkDc4tKNXywjsB_mfP3abpA1zDKWg"
FOLDER_ID = "1IxeQ04Wxk3AAurdxULC3TCCU-HDoOaVt"

bot = telebot.TeleBot(BOT_TOKEN)

# Authenticate Google Drive
json_key_path = "/content/drive/MyDrive/happyfarmbzubotproject-fca5f1cb527a.json"
scope = ["https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name(json_key_path, scope)
gauth = GoogleAuth()
gauth.credentials = creds
drive = GoogleDrive(gauth)

print("\u2705 Google Drive authentication successful!")

# Global storage for user data
user_choices = {}   # plant type
sensor_values = {}  # sensor JSON
user_languages = {} # Store user language preferences
latest_results = {}  # chat_id: result_text


def get_sensor_data_from_drive():
    try:
        # Create a file object by fileId (the fileId from your ESP code)
        sensor_file = drive.CreateFile({'id': '1nB0K5Qrhf3XcwT9_WpSg-4RhmfEnPNRq'})
        sensor_file.GetContentFile('sensorData.json')

        # Load the sensor data from the downloaded file
        with open('sensorData.json', 'r') as file:
            sensor_data = json.load(file)
        return sensor_data
    except Exception as e:
        print("❌ Error fetching sensor data:", e)
        return None

# --- Define Models and Disease Info ---


models = {
    "Lettuce": {
        "path": "/content/drive/MyDrive/Datasets/LettuceDataset/LettuceResults/best.pt",
        "classes": [
            "Bacterial", "Downy_mildew_on_lettuce", "Lettuce Mosaic Virus",
            "Powdery_mildew_on_lettuce", "Septoria_Blight_on_lettuce",
            "Wilt and leaf blight on lettuce", "healthy"
        ]
    },
    "Cucumber": {
        "path": "/content/drive/MyDrive/Datasets/CucumberDataset/best.pt",
        "classes": ["Anthracnose", "Bacterial Wilt", "Downy Mildew", "Fresh Leaf", "Gummy Stem Blight","Powdery Mildew"]
    },
    "Tomato": {
        "path": "/content/drive/MyDrive/Datasets/TomatoDataset/tomatoResult/bestTomato.pt",
        "classes": [ "Gray spot", "powdery mildew", "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___healthy", "Tomato___Late_blight", "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot"]
    }
}

disease_info = {
    "Bacterial": {
        "name_ar": "Bacterial-المرض البكتيري",
        "description_en": "A bacterial disease that causes yellowing, wilting, and necrosis in lettuce. 🌿⚠️",
        "cause_en": "Caused by bacterial pathogens such as Xanthomonas spp. or Pseudomonas spp. 🦠",
        "treatment_en": "Use copper-based fungicides, remove infected leaves, and avoid overhead watering. 💧❌",
        "description_ar": "مرض بكتيري يسبب الاصفرار والذبول والنخر في خس. 🌿⚠️",
        "cause_ar": "تسببه العوامل البكتيرية مثل Xanthomonas spp. أو Pseudomonas spp. 🦠",
        "treatment_ar": "استخدم المبيدات النحاسية، وازالة الأوراق المصابة، وتجنب الري من الأعلى. 💧❌"
    },
    "Downy_mildew_on_lettuce": {
        "name_ar": "Downy_mildew- العفن الزغبي على الخس",
        "description_en": "A fungal disease that causes yellowing and downy spots on the underside of lettuce leaves. 🍂🌱",
        "cause_en": "Caused by the oomycete pathogen Peronospora farinosa. 🍄",
        "treatment_en": "Use fungicides, improve air circulation, and remove infected plant parts. 🍃💨",
        "description_ar": "مرض فطري يسبب الاصفرار والبقع الفروية على الجانب السفلي لأوراق الخس. 🍂🌱",
        "cause_ar": "تسببه الفطريات العيشية Peronospora farinosa. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية، وحسن التهوية، وازالة الأجزاء المصابة من النبات. 🍃💨"
    },
    "Lettuce_Mosaic_Virus": {
        "name_ar": "فيروس موزاييك الخس",
        "description_en": "A viral disease causing mosaic-like symptoms on lettuce leaves. 🍃💔",
        "cause_en": "Caused by the Lettuce mosaic virus (LMV). 🦠",
        "treatment_en": "Remove infected plants and use resistant varieties. 🚫🌱",
        "description_ar": "مرض فيروسي يسبب أعراض تشبه الفسيفساء على أوراق الخس. 🍃💔",
        "cause_ar": "تسببه الفيروسات من نوع Lettuce mosaic virus (LMV). 🦠",
        "treatment_ar": "قم بإزالة النباتات المصابة واستخدم الأصناف المقاومة. 🚫🌱"
    },
    "Powdery_mildew_on_lettuce": {
        "name_ar": "البياض الدقيقي على الخس",
        "description_en": "A fungal disease that forms a white powdery coating on the tops of lettuce leaves. ❄️🍃",
        "cause_en": "Caused by the fungus Erysiphe cichoracearum. 🍄",
        "treatment_en": "Use fungicides and ensure proper spacing to avoid high humidity. 🌬️💧",
        "description_ar": "مرض فطري يشكل طبقة بيضاء بودرية على أعلى أوراق الخس. ❄️🍃",
        "cause_ar": "تسببه الفطريات Erysiphe cichoracearum. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وتأكد من المسافات المناسبة لتجنب الرطوبة العالية. 🌬️💧"
    },
    "Septoria_Blight_on_lettuce": {
        "name_ar": "اللفحة السبتيورية على الخس",
        "description_en": "A fungal disease causing lesions with dark edges on lettuce leaves. 🥬⚠️",
        "cause_en": "Caused by the fungus Septoria spp. 🍄",
        "treatment_en": "Remove affected leaves and apply fungicides. 🍂🔬",
        "description_ar": "مرض فطري يسبب التقرحات ذات الحواف الداكنة على أوراق الخس. 🥬⚠️",
        "cause_ar": "تسببه الفطريات من نوع Septoria spp. 🍄",
        "treatment_ar": "قم بإزالة الأوراق المصابة وطبق المبيدات الفطرية. 🍂🔬"
    },
    "Wilt_and_leaf_blight_on_lettuce": {
        "name_ar": "الذبول ولفحة الأوراق على الخس",
        "description_en": "A disease that causes wilting and browning of lettuce leaves. 🌿💔",
        "cause_en": "Caused by various pathogens, including Fusarium wilt and bacterial wilt. 🦠",
        "treatment_en": "Remove infected plants, apply appropriate fungicides, and improve soil drainage. 🌱💧",
        "description_ar": "مرض يسبب ذبول وتحول أوراق الخس إلى اللون البني. 🌿💔",
        "cause_ar": "تسببه عدة عوامل ممرضة بما في ذلك ذبول Fusarium وذبول بكتيري. 🦠",
        "treatment_ar": "قم بإزالة النباتات المصابة، وطبق المبيدات الفطرية المناسبة، وحسن تصريف التربة. 🌱💧"
    },
    "healthy": {
        "name_ar": "النبات سليم",
        "description_en": "The plant is healthy and free of any visible diseases. 🌱✅",
        "cause_en": "N/A",
        "treatment_en": "Continue normal care and monitoring. 🌞🌿",
        "description_ar": "النبات صحي وخالي من أي أمراض ظاهرة. 🌱✅",
        "cause_ar": "غير متاح",
        "treatment_ar": "استمر في العناية الطبيعية والمراقبة. 🌞🌿"
    },


    # Cucumber diseases

    "Anthracnose": {
        "name_ar": "الأنثراكنوز",
        "description_en": "A fungal disease that causes sunken lesions and dark spots on leaves and fruits. 🍈⚠️",
        "cause_en": "Caused by the fungus Colletotrichum spp. 🍄",
        "treatment_en": "Use fungicides, remove infected plants, and rotate crops to reduce spread. 🔄🦠",
        "description_ar": "مرض فطري يسبب تقرحات غائرة وبقع داكنة على الأوراق والفواكه. 🍈⚠️",
        "cause_ar": "تسببه الفطريات من نوع Colletotrichum spp. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية، وازالة النباتات المصابة، ودور المحاصيل للحد من الانتشار. 🔄🦠"
    },
    "Bacterial Wilt": {
        "name_ar": "الذبول البكتيري",
        "description_en": "A bacterial infection that causes yellowing, wilting, and rapid collapse of cucumber plants. 🌿⚠️",
        "cause_en": "Caused by the bacterium Erwinia tracheiphila. 🦠",
        "treatment_en": "Remove infected plants and use resistant cucumber varieties. ❌🌱",
        "description_ar": "عدوى بكتيرية تسبب اصفرار، ذبول، وانهيار سريع لنباتات الخيار. 🌿⚠️",
        "cause_ar": "تسببها البكتيريا Erwinia tracheiphila. 🦠",
        "treatment_ar": "قم بإزالة النباتات المصابة واستخدم أصناف الخيار المقاومة. ❌🌱"
    },
    "Downy Mildew": {
        "name_ar": "البياض الزغبي",
        "description_en": "A fungal disease that causes yellow spots on the upper leaf surfaces and a grayish mold underneath. 🍂🦠",
        "cause_en": "Caused by the fungus Pseudoperonospora cubensis. 🍄",
        "treatment_en": "Apply fungicides, remove affected leaves, and improve air circulation. 💨🌿",
        "description_ar": "مرض فطري يسبب بقع صفراء على الأسطح العليا للأوراق وعفن رمادي من الأسفل. 🍂🦠",
        "cause_ar": "تسببه الفطريات Pseudoperonospora cubensis. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية، وأزل الأوراق المصابة، وحسن التهوية. 💨🌿"
    },
    "Fresh Leaf": {
         "name_ar": "ورقة صحية",
        "description_en": "Healthy leaves that are free from disease. 🌱😊",
        "cause_en": "N/A",
        "treatment_en": "Continue proper care to maintain healthy plants. 🌿💧",
        "description_ar": "أوراق صحية خالية من الأمراض. 🌱😊",
        "cause_ar": "لا يوجد",
        "treatment_ar": "استمر في العناية السليمة للحفاظ على النباتات الصحية. 🌿💧"
    },
    "Gummy Stem Blight": {
        "name_ar": "لفحة الساق الصمغية",
        "description_en": "A fungal disease that causes lesions and gummy exudates on stems and leaves. 🌿💧",
        "cause_en": "Caused by the fungus Didymella bryoniae. 🍄",
        "treatment_en": "Use fungicides and remove infected plants to prevent spread. 🦠❌",
        "description_ar": "مرض فطري يسبب تقرحات وإفرازات لزجة على السيقان والأوراق. 🌿💧",
        "cause_ar": "تسببه الفطريات Didymella bryoniae. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وأزل النباتات المصابة لمنع الانتشار. 🦠❌"
    },
    "Powdery Mildew": {
        "name_ar": "البياض الدقيقي",
        "description_en": "A fungal disease that appears as white, powdery spots on leaves, stems, and sometimes fruits. 🍂🌨️",
        "cause_en": "Caused by the fungus Podosphaera xanthii. 🍄",
        "treatment_en": "Apply fungicides and ensure good air circulation by proper plant spacing. 💨🌱",
        "description_ar": "مرض فطري يظهر على شكل بقع بيضاء ومسحوقة على الأوراق والسيقان وأحيانًا الفواكه. 🍂🌨️",
        "cause_ar": "تسببه الفطريات Podosphaera xanthii. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وتأكد من توفير التهوية الجيدة عن طريق المسافات المناسبة بين النباتات. 💨🌱"
    },

    # Tomato diseases
    "Gray spot": {
        "name_ar": "البقعة الرمادية",
        "description_en": "A fungal disease that causes grayish-brown lesions on tomato leaves and stems. 🍅🦠",
        "cause_en": "Caused by the fungus Alternaria solani. 🍄",
        "treatment_en": "Use fungicides and practice crop rotation to avoid reinfection. 🔄🌿",
        "description_ar": "مرض فطري يسبب تقرحات بنيّة رمادية على أوراق وسيقان الطماطم. 🍅🦠",
        "cause_ar": "تسببه الفطريات Alternaria solani. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية واتباع دورة المحاصيل لتجنب التلوث المتكرر. 🔄🌿"
    },
    "Powdery mildew": {
        "name_ar": "البياض الدقيقي",
        "description_en": "A fungal disease that causes a white powdery coating on the upper side of tomato leaves. 🍅🌨️",
        "cause_en": "Caused by the fungus Leveillula taurica. 🍄",
        "treatment_en": "Use fungicides, improve air circulation, and avoid overhead watering. 💨💧",
        "description_ar": "مرض فطري يسبب تغطية بيضاء مسحوقية على الجانب العلوي من أوراق الطماطم. 🍅🌨️",
        "cause_ar": "تسببه الفطريات Leveillula taurica. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية، وحسن التهوية، وتجنب الري من الأعلى. 💨💧"
    },
    "Tomato___Bacterial_spot": {
        "name_ar": "البقعة البكتيرية",
        "description_en": "A bacterial disease that causes small, dark lesions on tomato leaves, stems, and fruit. 🍅🦠",
        "cause_en": "Caused by Xanthomonas spp. 🦠",
        "treatment_en": "Use copper-based fungicides and remove infected plant parts. ❌🌱",
        "description_ar": "مرض بكتيري يسبب تقرحات صغيرة داكنة على أوراق وسيقان وفواكه الطماطم. 🍅🦠",
        "cause_ar": "تسببه بكتيريا Xanthomonas spp. 🦠",
        "treatment_ar": "استخدم المبيدات الفطرية المعتمدة على النحاس وأزل أجزاء النبات المصابة. ❌🌱"
    },
    "Tomato___Early_blight": {
        "name_ar": "اللفحة المبكرة",
        "description_en": "A fungal disease that causes dark, concentric rings on leaves and stems. 🍅🦠",
        "cause_en": "Caused by the fungus Alternaria solani. 🍄",
        "treatment_en": "Use fungicides, remove infected leaves, and practice crop rotation. 🔄🌿",
        "description_ar": "مرض فطري يسبب دوائر داكنة على أوراق وسيقان الطماطم. 🍅🦠",
        "cause_ar": "تسببه الفطريات Alternaria solani. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية، وأزل الأوراق المصابة، واتباع دورة المحاصيل. 🔄🌿"
    },
    "Tomato___healthy": {
        "name_ar": "نبات صحي",
        "description_en": "The plant is healthy with no visible signs of disease. 🍅😊",
        "cause_en": "N/A",
        "treatment_en": "Maintain proper care with regular watering and sunlight. 💧🌞",
        "description_ar": "النبات صحي ولا توجد علامات على وجود مرض. 🍅😊",
        "cause_ar": "لا يوجد",
        "treatment_ar": "حافظ على العناية السليمة مع الري المنتظم وأشعة الشمس. 💧🌞"
    },
    "Tomato___Late_blight": {
        "name_ar": "اللفحة المتأخرة",
        "description_en": "A fungal disease that causes dark, water-soaked lesions on leaves and stems. 🍅🌧️",
        "cause_en": "Caused by the fungus Phytophthora infestans. 🍄",
        "treatment_en": "Use fungicides and remove infected plant parts. Ensure proper spacing to reduce humidity. ❌💧",
        "description_ar": "مرض فطري يسبب تقرحات داكنة ومشبعة بالماء على أوراق وسيقان الطماطم. 🍅🌧️",
        "cause_ar": "تسببه الفطريات Phytophthora infestans. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وأزل أجزاء النبات المصابة. تأكد من التباعد المناسب لتقليل الرطوبة. ❌💧"
    },
    "Tomato___Leaf_Mold": {
        "name_ar": "عفن الأوراق",
        "description_en": "A fungal disease that causes yellowing and mold growth on the undersides of tomato leaves. 🍅🍂",
        "cause_en": "Caused by the fungus Passalora fulva. 🍄",
        "treatment_en": "Apply fungicides and remove affected leaves. 💨🌿",
        "description_ar": "مرض فطري يسبب اصفرار ونمو العفن على الجوانب السفلية لأوراق الطماطم. 🍅🍂",
        "cause_ar": "تسببه الفطريات Passalora fulva. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وأزل الأوراق المصابة. 💨🌿"
    },
    "Tomato___Septoria_leaf_spot": {
        "name_ar": "بقعة الأوراق السبتيورية",
        "description_en": "A fungal disease that causes dark spots with a yellow halo on tomato leaves. 🍅🌑",
        "cause_en": "Caused by the fungus Septoria lycopersici. 🍄",
        "treatment_en": "Use fungicides and remove affected leaves. ❌🍃",
        "description_ar": "مرض فطري يسبب بقع داكنة مع هالة صفراء على أوراق الطماطم. 🍅🌑",
        "cause_ar": "تسببه الفطريات Septoria lycopersici. 🍄",
        "treatment_ar": "استخدم المبيدات الفطرية وأزل الأوراق المصابة. ❌🍃"
    }
}

disease_causes = {
    # Lettuce Diseases
    "Bacterial": {
        "causes": ["High air humidity", "Poor air circulation", "High soil humidity"],
        "sensor_keys": {
            "air_humidity": "high",
            "rs_hum": "high"
        }
    },
    "Downy_mildew_on_lettuce": {
        "causes": ["High air humidity", "Dark lighting", "Low soil temperature"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low",
            "rs_temp": "low"
        }
    },
    "Lettuce Mosaic Virus": {
        "causes": ["Infected soil pH", "Poor soil conductivity"],
        "sensor_keys": {
            "rs_ph": "high",
            "rs_cond": "low"
        }
    },
    "Powdery_mildew_on_lettuce": {
        "causes": ["High air humidity", "Low light"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low"
        }
    },
    "Septoria_Blight_on_lettuce": {
        "causes": ["High air humidity", "Low soil pH", "Dark lighting"],
        "sensor_keys": {
            "air_humidity": "high",
            "rs_ph": "low",
            "ldr_d": "low"
        }
    },
    "Wilt and leaf blight on lettuce": {
        "causes": ["Low soil humidity", "High air temperature", "Low soil conductivity"],
        "sensor_keys": {
            "rs_hum": "low",
            "temperature": "high",
            "rs_cond": "low"
        }
    },

    # Cucumber Diseases
    "Anthracnose": {
        "causes": ["High air humidity", "High air temperature", "Low soil pH"],
        "sensor_keys": {
            "air_humidity": "high",
            "temperature": "high",
            "rs_ph": "low"
        }
    },
    "Bacterial Wilt": {
        "causes": ["High air temperature", "Low soil humidity"],
        "sensor_keys": {
            "temperature": "high",
            "rs_hum": "low"
        }
    },
    "Downy Mildew": {
        "causes": ["High air humidity", "Dark lighting", "Low soil temperature"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low",
            "rs_temp": "low"
        }
    },
    "Fresh Leaf": {
        "causes": [],
        "sensor_keys": {}
    },
    "Gummy Stem Blight": {
        "causes": ["High soil humidity", "Low soil pH"],
        "sensor_keys": {
            "rs_hum": "high",
            "rs_ph": "low"
        }
    },
    "Powdery Mildew": {
        "causes": ["High air humidity", "Low light", "Poor air circulation"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low"
        }
    },

    # Tomato Diseases
    "Gray spot": {
        "causes": ["High air humidity", "Dark light conditions", "Low soil EC"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low",
            "rs_cond": "low"
        }
    },
    "powdery mildew": {
        "causes": ["High air humidity", "Low light", "Poor air movement"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low"
        }
    },
    "Tomato___Bacterial_spot": {
        "causes": ["High air humidity", "Low soil pH"],
        "sensor_keys": {
            "air_humidity": "high",
            "rs_ph": "low"
        }
    },
    "Tomato___Early_blight": {
        "causes": ["High soil humidity", "High temperature", "Low soil EC"],
        "sensor_keys": {
            "rs_hum": "high",
            "temperature": "high",
            "rs_cond": "low"
        }
    },
    "Tomato___healthy": {
        "causes": [],
        "sensor_keys": {}
    },
    "Tomato___Late_blight": {
        "causes": ["High air humidity", "Low soil temperature", "Poor lighting"],
        "sensor_keys": {
            "air_humidity": "high",
            "rs_temp": "low",
            "ldr_d": "low"
        }
    },
    "Tomato___Leaf_Mold": {
        "causes": ["High air humidity", "Dark light", "High soil humidity"],
        "sensor_keys": {
            "air_humidity": "high",
            "ldr_d": "low",
            "rs_hum": "high"
        }
    },
    "Tomato___Septoria_leaf_spot": {
        "causes": ["High air humidity", "Low soil pH"],
        "sensor_keys": {
            "air_humidity": "high",
            "rs_ph": "low"
        }
    }
}



#------------------------------LANGUAGE-------------------------------------------------

@bot.callback_query_handler(func=lambda call: call.data in ["/language", "/tips", "/ChoosePlant", "/sensors"])
def simulate_command(call):
    chat_id = call.message.chat.id

    if call.data == "/language":
        # Simulate /language command
        choose_language(call.message)

    elif call.data == "/tips":
        # Simulate /tips command
        handle_tips(call.message)

    elif call.data == "/ChoosePlant":
        # Simulate /ChoosePlant command

        handle_capture(call.message)  # Make sure you have a command handler named choose_plant

    elif call.data == "/sensors":
        handle_sensors(call)  # Your existing /sensors command handler

    bot.answer_callback_query(call.id)  # Avoid "loading" spinner






@bot.message_handler(commands=['language'])
def choose_language(message):
    chat_id = message.chat.id
    markup = InlineKeyboardMarkup()
    markup.row(
        InlineKeyboardButton(" English", callback_data='lang_en'),
        InlineKeyboardButton(" العربية", callback_data='lang_ar')
    )
    bot.send_message(chat_id, "🌐 Please choose your language:\nيرجى اختيار لغتك:", reply_markup=markup)


@bot.callback_query_handler(func=lambda call: call.data.startswith('lang_'))
def handle_language_choice(call):
    chat_id = call.message.chat.id
    language_code = call.data.split('_')[1]

    user_languages[chat_id] = language_code  # Save user choice

    if language_code == 'en':
        text = "✅ Language set to *English*.\n\nClick the button below to begin! 🌱"
        button_text = "📊 Start Smart Farming"
    else:
        text = "✅ تم اختيار اللغة *العربية*.\n\nاضغط على الزر أدناه للبدء! 🌿"
        button_text = "📊 ابدأ الزراعة الذكية"

    # Create the inline keyboard with a button that triggers /start
    keyboard = types.InlineKeyboardMarkup()
    keyboard.add(types.InlineKeyboardButton(button_text, callback_data='/start'))

    bot.send_message(chat_id, text, reply_markup=keyboard, parse_mode="Markdown")



def get_plant_type_translations(plant_type):
    plant_type_translation = {
        "lettuce": "خس",
        "tomato": "طماطم",
        "cucumber": "خيار"

    }
    plant_type_en = plant_type.capitalize()
    plant_type_ar = plant_type_translation.get(plant_type.lower(), plant_type_en)
    return plant_type_en, plant_type_ar

#------------------------------ START -------------------------------------------------
@bot.message_handler(commands=['start'])
def handle_start(message):
    chat_id = message.chat.id
    handle_start_from_callback(chat_id)

# 👉 Callback when user clicks 📊 Start Smart Farming
@bot.callback_query_handler(func=lambda call: call.data == "/start")
def handle_start_callback(call):
    chat_id = call.message.chat.id
    handle_start_from_callback(chat_id)


# 👉 Shared function for start logic (used by both methods above)
def handle_start_from_callback(chat_id):

    # If user has not selected a language yet
    if chat_id not in user_languages:
        markup = InlineKeyboardMarkup()
        markup.row(
            InlineKeyboardButton(" English", callback_data='lang_en'),
            InlineKeyboardButton(" العربية", callback_data='lang_ar')
        )
        bot.send_message(
            chat_id,
            "🌐 Please choose your language first:\nيرجى اختيار لغتك أولاً:",
            reply_markup=markup
        )
        return

    lang = user_languages.get(chat_id, 'en')

    if lang == 'ar':
        welcome_message = """
👋 *مرحبًا بك في مساعد الزراعة الذكية!* 🌿

يساعدك هذا البوت في:
✅ كشف أمراض أوراق النباتات باستخدام الذكاء الاصطناعي
✅ مراقبة بيانات الحساسات البيئية في الوقت الفعلي
✅ الحصول على نصائح رعاية مخصصة للمحاصيل التي تختارها

🔧 *طريقة الاستخدام:*
1️⃣ اضغط على زر *اختيار المحصول* أدناه لاختيار نوع المحصول (طماطم، خس، أو خيار).
2️⃣ أرسل صورة واضحة ومضاءة جيدًا لورقة النبات.
3️⃣ سيقوم البوت بتحليل الصورة وتقديم التشخيص المناسب.

📊 بعد اختيار المحصول، يمكنك:
— الضغط على زر *بيانات الحساسات* للاطلاع على القراءات البيئية الفعلية.
— الضغط على زر *الحصول على نصائح* للحصول على توصيات رعاية مفصلة.
🌐 لتغيير اللغة، استخدم زر *تغيير اللغة*.

💡 *نصيحة:* جودة الصورة العالية تؤدي إلى نتائج تحليل أكثر دقة.

🚀 ابدأ الآن باستخدام زر *اختر المحصول* أدناه!
"""
        keyboard = InlineKeyboardMarkup(row_width=2)
        keyboard.add(
            InlineKeyboardButton("🌱 اختر المحصول", callback_data="/ChoosePlant"),
            InlineKeyboardButton("🌐 تغيير اللغة", callback_data="/language")
        )
    else:
        welcome_message = """
👋 *Welcome to the Smart Farming Plant Assistant!* 🌿

This bot is designed to help you:
✅ Detect plant leaf diseases using AI
✅ Monitor real-time environmental sensor data
✅ Receive personalized care tips for your selected crops

🔧 *How It Works:*
1️⃣ Click the *Choose Plant* button below to select a crop (tomato, lettuce, or cucumber).
2️⃣ Send a clear, well-lit image of the plant's leaf.
3️⃣ The bot will analyze the image and provide a diagnosis.

📊 After selecting a plant, you can:
— Tap the *Sensor Data* button to check real-time environmental conditions.
— Tap the *Get Tips* button to receive tailored care recommendations.
🌐 To change the language, use the *Change Language* button.

💡 *Tip:* High-quality images result in more accurate analysis.

🚀 Get started by using the *Choose Plant* button below!
"""
        keyboard = InlineKeyboardMarkup(row_width=2)
        keyboard.add(
            InlineKeyboardButton("🌱 Choose Plant", callback_data="/ChoosePlant"),
            InlineKeyboardButton("🌐 Change Language", callback_data="/language")
        )

    bot.send_message(chat_id, welcome_message, reply_markup=keyboard, parse_mode="Markdown")
#----------------------CHOOSE PLANT-----------------------------------------------

@bot.message_handler(commands=['ChoosePlant'])
def handle_capture(message):
    chat_id = message.chat.id
    lang = user_languages.get(chat_id, "en")
    user_choices.pop(chat_id, None)

    if lang == "ar":
        text = "🌱 *اختر نوع النبات الذي ترغب في تحليله:*"
    else:
        text = "🌱 *Please select the plant you want to analyze:*"

    bot.send_message(chat_id, text, reply_markup=plant_selection_markup(chat_id), parse_mode="Markdown")



def plant_selection_markup(chat_id):
    lang = user_languages.get(chat_id, 'en')
    markup = InlineKeyboardMarkup()

    if lang == 'ar':
        markup.add(InlineKeyboardButton("  خس 🥬", callback_data="Lettuce"))
        markup.add(InlineKeyboardButton(" خيار 🥒", callback_data="Cucumber"))
        markup.add(InlineKeyboardButton(" طماطم 🍅", callback_data="Tomato"))
    else:
        markup.add(InlineKeyboardButton("Lettuce 🥬", callback_data="Lettuce"))
        markup.add(InlineKeyboardButton("Cucumber 🥒", callback_data="Cucumber"))
        markup.add(InlineKeyboardButton("Tomato 🍅", callback_data="Tomato"))

    return markup


@bot.callback_query_handler(func=lambda call: call.data in ["Lettuce", "Cucumber", "Tomato"])
def handle_plant_selection(call):
    chat_id = call.message.chat.id
    user_choices[chat_id] = call.data

    lang = user_languages.get(chat_id, 'en')  # Get the user's language (default to English)

    # Get translated plant names for display
    plant_type_en, plant_type_ar = get_plant_type_translations(call.data.lower())

    # Create inline keyboard based on language
    if lang == "ar":
        keyboard = InlineKeyboardMarkup(row_width=2)
        keyboard.add(
            InlineKeyboardButton("🌱 عرض بيانات الحساسات", callback_data="/sensors"),
            InlineKeyboardButton("💡 الحصول على نصائح", callback_data="/tips"),
            InlineKeyboardButton("🔁 اختيار نبتة أخرى", callback_data="/ChoosePlant")
        )
        bot.send_message(
            chat_id,
            f"<b>🔹 لقد اخترت: {plant_type_ar} . الآن، أرسل صورة واضحة للورقة لإجراء التحليل، أو اضغط على أحد الأزرار أدناه:</b>",
            reply_markup=keyboard,
            parse_mode="HTML"
        )
    else:
        keyboard = InlineKeyboardMarkup(row_width=2)
        keyboard.add(
            InlineKeyboardButton("🌱 View Sensor Data", callback_data="/sensors"),
            InlineKeyboardButton("💡 Get Tips", callback_data="/tips"),
            InlineKeyboardButton("🔁 Choose Another Plant", callback_data="/ChoosePlant")

        )
        bot.send_message(
            chat_id,
            f"<b>🔹 You selected: {plant_type_en}. Now, send a clear leaf image for analysis, or tap one of the buttons below:</b>",
            reply_markup=keyboard,
            parse_mode="HTML"
        )


#------------------------------SENSORS NORMAL RANGE -------------------------------------------------


# Place utility function for range checking at the top of your code
def check_sensor_range(value, min_val, max_val):
    return min_val <= value <= max_val



# Conversion from mS/cm to dS/m
def convert_ec_to_ds(value_ms_cm):
    return value_ms_cm / 10  # Conversion factor from mS/cm to dS/m

# Function to fetch crop-specific data
def get_crop_info(crop_name):
    crop_data = {
        "tomato": {
            "growth_period": "60–85 days",
            "ranges": {
                "Air Temperature (°C)": (21, 29),
                "Air Humidity (%)": (60, 75),
                "Soil pH": (5.5, 7.5),
                "Soil EC (dS/m)": (2, 5),
                "Light (LDR Analog)": (100, 500),
                "Soil Temperature (°C)": (21.1, 35),

                "Soil Humidity (%)": (60, 80),
            }
        },
        "lettuce": {
            "growth_period": "65–130 days",
            "ranges": {
                "Air Temperature (°C)": (16, 18),
                "Air Humidity (%)": (50, 70),
                "Soil pH": (6.0, 7),
                "Soil EC (dS/m)": (1.2, 2),
                "Light (LDR Analog)": (500, 1500),
                "Soil Temperature (°C)": (4.4, 26.7),

                "Soil Humidity (%)": (40, 60),
            }
        },
        "cucumber": {
            "growth_period": "50–80 days",
            "ranges": {
                "Air Temperature (°C)": (16, 32),
                "Air Humidity (%)": (70, 85),
                "Soil pH": (5.5, 7.5),
                "Soil EC (dS/m)": (1.7, 2.5),
                "Light (LDR Analog)": (100, 500),
                "Soil Temperature (°C)": (15.6, 35),

                "Soil Humidity (%)": (65, 80),
            }
        }
    }
    return crop_data.get(crop_name.lower())

#-------------------------- SENSOR FEEDBACK ----------------------------------------------------

def get_sensor_feedback(sensor_values, crop_name, language="en"):
    crop_info = get_crop_info(crop_name)
    feedback_lines = []

    if crop_info is None:
        return (
            "⚠️ Unknown crop selected." if language == "en" else "⚠️ لم يتم التعرف على المحصول."
        )

    optimal_ranges = crop_info["ranges"]
    out_of_range_count = 0

    # Sensor name translations for Arabic
    sensor_translations = {
        "Air Temperature (°C)": "درجة حرارة الهواء (س°)",
        "Air Humidity (%)": "رطوبة الهواء (%)",
        "Light (LDR Analog)": "قيمة الضوء ",
        "Light (LDR Digital)": "شدة الضوء ",
        "Soil Temperature (°C)": "درجة حرارة التربة (س°)",
        "Soil Humidity (%)": "رطوبة التربة (%)",
        "Soil EC (dS/m)": "التوصيلية الكهربائية (EC)",
        "Soil pH": "مستوى الحموضة (pH)"
    }

    # Feedback emojis
    good_emoji = "✅"
    warn_emoji = "⚠️"
    up_arrow = "🔺"
    down_arrow = "🔻"

    for key, (min_val, max_val) in optimal_ranges.items():
        value = sensor_values.get(key)

        # Convert EC units if needed
        if key == "Soil EC (dS/m)":
            value = convert_ec_to_ds(value)

        # Only process numeric values
        try:
            numeric_value = float(value)
        except:
            continue

        in_range = check_sensor_range(numeric_value, min_val, max_val)

        if not in_range:
            out_of_range_count += 1
            direction = up_arrow if numeric_value > max_val else down_arrow

            # Use correct language key name
            display_key = sensor_translations.get(key, key) if language == "ar" else key

            # Calculate adjustment
            if numeric_value > max_val:
                adjustment_amount = numeric_value - max_val
                adjustment = (
                    f"Try decreasing the {display_key} by {adjustment_amount:.2f}°C to be within the optimal range.\n"
                    if language == "en"
                    else f"حاول تقليل {display_key} بمقدار {adjustment_amount:.2f} س° ليكون ضمن النطاق المثالي.\n"
                )
            elif numeric_value < min_val:
                adjustment_amount = min_val - numeric_value
                adjustment = (
                    f"Try increasing the {display_key} by {adjustment_amount:.2f}°C to be within the optimal range.\n"
                    if language == "en"
                    else f"حاول زيادة {display_key} بمقدار {adjustment_amount:.2f}س° ليكون ضمن النطاق المثالي.\n"
                )

            feedback_lines.append(
                f"{warn_emoji} *{display_key}* is out of range ({min_val}–{max_val}). Current: {numeric_value} {direction}. {adjustment}\n"
                if language == "en"
                else f"{warn_emoji} *{display_key}* خارج النطاق ({min_val}–{max_val}). الحالي: {numeric_value} {direction}. {adjustment}\n"
            )
        else:
            display_key = sensor_translations.get(key, key) if language == "ar" else key
            feedback_lines.append(
                f"{good_emoji} *{display_key}* is within optimal range.\n"
                if language == "en"
                else f"{good_emoji} *{display_key}* ضمن النطاق المثالي.\n"
            )

    # Summary at the top
    total = len(optimal_ranges)
    if language == "ar":
        summary = (
            f"{warn_emoji} {out_of_range_count} من أصل {total} من الحساسات خارج النطاق.\n\n"
            if out_of_range_count > 0
            else f"{good_emoji} جميع قراءات الحساسات ضمن النطاق المثالي لمحصول {crop_name.lower()}.\n\n"
        )
    elif language == "en":
        summary = (
            f"{warn_emoji} {out_of_range_count} out of {total} sensors are out of range.\n\n"
            if out_of_range_count > 0
            else f"{good_emoji} All sensor readings are within optimal ranges for {crop_name.lower()}.\n\n"
        )
    else:
        summary = (
            f"{warn_emoji} {out_of_range_count}/{total} sensors are not in range.\n\n"
            if out_of_range_count > 0
            else f"{good_emoji} All sensors are in healthy range for {crop_name.lower()}.\n\n"
        )

    return summary + "\n".join(feedback_lines)



#----------------------- ADVICE ----------------------


def generate_disease_sensor_advice(disease_name, sensor_values, crop_name, language="en"):
    crop_info = get_crop_info(crop_name)
    if not crop_info:
        return ""

    optimal_ranges = crop_info["ranges"]
    disease_info = disease_causes.get(disease_name)
    if not disease_info:
        return ""

    sensor_keys = disease_info.get("sensor_keys", {})
    sensor_feedback = []

    # Map internal sensor keys to readable names
    sensor_label_map_en = {
        "temperature": "Air Temperature (°C)",
        "air_humidity": "Air Humidity (%)",
        "rs_ph": "Soil pH",
        "rs_cond": "Soil EC (dS/m)",
        "ldr_d": "Light (LDR Analog)",
        "rs_temp": "Soil Temperature (°C)",
        "rs_hum": "Soil Humidity (%)"
    }


    sensor_label_map_ar = {
        "temperature": "درجة حرارة الهواء (°C)",
        "air_humidity": "رطوبة الهواء (%)",
        "rs_ph": "درجة حموضة التربة",
        "rs_cond": "توصيل التربة الكهربائي (dS/m)",
        "ldr_d": "الضوء (LDR)",
        "rs_temp": "درجة حرارة التربة (°C)",
        "rs_hum": "رطوبة التربة (%)"
    }

# Choose correct label map based on language
    sensor_label_map = sensor_label_map_ar if language == "ar" else sensor_label_map_en


    cause_phrases_en = {
        "low": "Too low — consider increasing it.",
        "high": "Too high — consider decreasing it."
    }

    cause_phrases_ar = {
        "low": "منخفض جدًا — يُفضل زيادته.",
        "high": "مرتفع جدًا — يُفضل تقليله."
    }

    for key, expected_direction in sensor_keys.items():
        readable_key = sensor_label_map.get(key)
        english_key = sensor_label_map_en.get(key)
        if not readable_key:
            continue

        sensor_val = sensor_values.get(sensor_label_map_en.get(key))  # Always use EN keys to get sensor value

        if key == "rs_cond":
            sensor_val = convert_ec_to_ds(sensor_val)

        try:
            value = float(sensor_val)
        except:
            continue

        min_val, max_val = optimal_ranges.get(english_key, (None, None))
        if min_val is None or max_val is None:
            continue

        if value < min_val:
            direction = "low"
        elif value > max_val:
            direction = "high"
        else:
            continue  # In range

        if direction != expected_direction:
            continue  # Skip this sensor — not matching disease cause

        phrase = cause_phrases_ar[direction] if language == "ar" else cause_phrases_en[direction]
        if language == "ar":
            sensor_feedback.append(f"➤ *{readable_key} ({sensor_label_map_en.get(key)})* خارج النطاق المثالي. {phrase}")
        else:
            sensor_feedback.append(f"➤ *{readable_key}* is out of optimal range. {phrase}")

    if sensor_feedback:
        header = "\n\n🌿 *Possible sensor-related causes for this disease:*\n" if language == "en" else "\n\n🌿 *الأسباب المحتملة المتعلقة بالحساسات لهذا المرض:*\n"
        return header + "\n".join(sensor_feedback)
    else:
        no_advice_msg = (
            "\n\n✅ Based on current sensor readings, no environmental factors related to this disease have been detected."
            if language == "en"
            else "\n\n✅ بناءً على قراءات الحساسات الحالية، لم يتم اكتشاف أي عوامل بيئية مرتبطة بهذا المرض."
        )
        return no_advice_msg

#----------------------Sensors---------------------------------------------------------------------------------

def format_sensor_message(plant_type, sensor_data, feedback, lang="en"):
    # Check the LDR digital value and set the display text
    ldr_digital_status = "Bright" if sensor_data['Light (LDR Digital)'] == 0 else "Dark"

    if lang == "ar":
        return (
            f"*📥 تم استلام بيانات الحساس لمحصول {plant_type}:*\n"
            f"- 🌡️ درجة حرارة الهواء: {sensor_data['Air Temperature (°C)']} °س\n"
            f"- 💧 رطوبة الهواء: {sensor_data['Air Humidity (%)']} %\n"
            f"- 💡 قيمة الضوء : {sensor_data['Light (LDR Analog)']}\n"
            f"- 💡 شدة الضوء : {ldr_digital_status}\n\n"
            f"- 🌡️ درجة حرارة التربة: {sensor_data['Soil Temperature (°C)']} °س\n"
            f"- 💧 رطوبة التربة: {sensor_data['Soil Humidity (%)']} %\n"
            f"- ⚡ التوصيلية الكهربائية (EC): {sensor_data['Soil EC (dS/m)']} dS/m\n"
            f"- 🧪 مستوى الحموضة (pH): {sensor_data['Soil pH']}\n\n"
            f"*🌱 نتائج التحليل:*\n{feedback}"
        )
    else:
        return (
            f"*📥 Sensor Data Received for {plant_type}:*\n"
            f"- 🌡️ Air Temperature: {sensor_data['Air Temperature (°C)']} °C\n"
            f"- 💧 Air Humidity: {sensor_data['Air Humidity (%)']} %\n"
            f"- 💡 Light (Analog): {sensor_data['Light (LDR Analog)']}\n"
            f"- 💡 Light (Digital): {ldr_digital_status}\n\n"
            f"- 🌡️ Soil Temperature: {sensor_data['Soil Temperature (°C)']} °C\n"
            f"- 💧 Soil Humidity: {sensor_data['Soil Humidity (%)']} %\n"
            f"- ⚡ Conductivity (EC): {sensor_data['Soil EC (dS/m)']} dS/m\n"
            f"- 🧪 pH Level: {sensor_data['Soil pH']}\n\n"
            f"*🌱 Sensor Feedback:*\n{feedback}"
        )


# Main /sensors command handler

def handle_sensors(call):
    chat_id = call.message.chat.id
    lang = user_languages.get(chat_id, 'en')

    if chat_id not in user_choices:
        if lang == "ar":
            bot.send_message(chat_id, "⚠️ يرجى اختيار نوع النبات أولاً باستخدام /ChoosePlant قبل استخدام /sensors.")
        else:
            bot.send_message(chat_id, "⚠️ Please select a plant type first using /ChoosePlant before using /sensors.")
        return

    # Fetch sensor data from Drive (same as no-payload case)
    sensor_data = get_sensor_data_from_drive()
    if sensor_data is None:
        if lang == "ar":
            bot.send_message(chat_id, "❌ لم يتمكن من جلب بيانات الحساس من Google Drive.")
        else:
            bot.send_message(chat_id, "❌ Could not fetch sensor data from Google Drive.")
        return

    # Use the selected plant type (pop it to enforce one-time use)
    #plant_type = user_choices.pop(chat_id)
    plant_type = user_choices.get(chat_id)


    # Extract sensor values as before
    temperature = sensor_data.get('temperature', 'N/A')
    humidity = sensor_data.get('humidity', 'N/A')
    ldr_a = sensor_data.get('ldrAnalog', 'N/A')
    ldr_d = sensor_data.get('ldrDigital', 'N/A')
    rs_temp = sensor_data.get('rs485_temperature', 'N/A')
    rs_hum = sensor_data.get('rs485_humidity', 'N/A')
    rs_cond = sensor_data.get('rs485_conductivity', 'N/A')
    rs_ph = sensor_data.get('rs485_ph', 'N/A')

    sensor_values[chat_id] = {
        "Air Temperature (°C)": temperature,
        "Air Humidity (%)": humidity,
        "Light (LDR Analog)": ldr_a,
        "Light (LDR Digital)": ldr_d,
        "Soil Temperature (°C)": rs_temp,
        "Soil Humidity (%)": rs_hum,
        "Soil EC (dS/m)": rs_cond,
        "Soil pH": rs_ph
    }

    sensor_for_feedback = sensor_values[chat_id]

    sensor_feedback = get_sensor_feedback(sensor_for_feedback, plant_type, lang)

    message_text = format_sensor_message(plant_type, sensor_for_feedback, sensor_feedback, lang)

    bot.send_message(chat_id, message_text, parse_mode="Markdown")

    # Show next action buttons as you do in handle_sensors()
    keyboard = types.InlineKeyboardMarkup()
    if lang == "ar":
        keyboard.add(types.InlineKeyboardButton("🔁 تحليل صورة ", callback_data="/photo"))
        keyboard.add(types.InlineKeyboardButton("💡 احصل على نصيحة", callback_data="/tips"))
        keyboard.add(types.InlineKeyboardButton("🌐 تغيير اللغة", callback_data="/language"))
        keyboard.add(types.InlineKeyboardButton("🌱 عرض بيانات الحساس", callback_data="/sensors"))
        keyboard.add(types.InlineKeyboardButton("🔁 اختيار نبتة أخرى", callback_data="/ChoosePlant"))

        message = "🔄 ماذا ترغب في القيام به بعد ذلك؟"
    else:
        keyboard.add(types.InlineKeyboardButton("🔁 Analyze Image", callback_data="/photo"))
        keyboard.add(types.InlineKeyboardButton("💡 Get Tips", callback_data="/tips"))
        keyboard.add(types.InlineKeyboardButton("🌐 Change My Language", callback_data="/language"))
        keyboard.add(types.InlineKeyboardButton("🌱 View Sensor Data", callback_data="/sensors"))
        keyboard.add(types.InlineKeyboardButton("🔁 Choose Another Plant", callback_data="/ChoosePlant"))

        message = "🔄 What would you like to do next?"

    bot.send_message(chat_id, message, reply_markup=keyboard)


#---------------------- IMAGE ANALYSIS ---------------------------------------------------------------------------------------


def format_disease_name(name):
    # Replace underscores with spaces and capitalize each word
    return ' '.join(word.capitalize() for word in name.replace('_', ' ').split())

def translate_growth_stage(growth_stage, lang):
    if lang == "ar":
        return growth_stage.replace("days", "أيام")
    return growth_stage



@bot.callback_query_handler(func=lambda call: call.data == "/photo")
def handle_photo_button(call):
    chat_id = call.message.chat.id
    lang = user_languages.get(chat_id, 'en')

    # Prompt user to send a photo
    if lang == "ar":
        bot.send_message(chat_id, "*يرجى إرسال صورة واضحة للورقة لتحليلها.*", parse_mode="Markdown")
    else:
        bot.send_message(chat_id, "*Please send a clear leaf photo for analysis.*", parse_mode="Markdown")

    bot.answer_callback_query(call.id)


@bot.message_handler(content_types=['photo'])
def handle_image(message):
    chat_id = message.chat.id
    lang = user_languages.get(chat_id, 'en')



    if chat_id not in user_choices:
        bot.reply_to(message, "❌ Please select a plant type first using /ChoosePlant.")
        return

    if not message.photo:
        msg = "❌ الرجاء إرسال صورة صالحة." if lang == "ar" else "❌ Please send a valid image."
        bot.reply_to(message, msg)
        return


    plant_type = user_choices[chat_id]
    plant_type_en, plant_type_ar = get_plant_type_translations(plant_type)
    model_info = models.get(plant_type)
    model = YOLO(model_info["path"])
    class_names = model_info["classes"]

    # Download and save the image
    file_id = message.photo[-1].file_id
    file_info = bot.get_file(file_id)
    file_url = f"https://api.telegram.org/file/bot{BOT_TOKEN}/{file_info.file_path}"
    img_path = f"/content/drive/My Drive/Telegram_Images/{plant_type}_image.jpg"
    with open(img_path, "wb") as f:
        f.write(requests.get(file_url).content)


    bot.send_message(chat_id, "🕵️‍♂️ جارٍ تحليل الصورة، يرجى الانتظار..." if lang == "ar" else "🕵️‍♂️ Analyzing the image, please wait...")


    try:
        results = model(img_path)
        pred_boxes = results[0].boxes

        if len(pred_boxes) > 0:
            pred_label = int(pred_boxes.cls[0].item())
            confidence = float(pred_boxes.conf[0].item()) * 100
            #predicted_disease = class_names[pred_label]

            raw_disease = class_names[pred_label]
            predicted_disease = raw_disease

            info = disease_info.get(predicted_disease, {})
            pretty_disease_name_en = format_disease_name(predicted_disease)
            pretty_disease_name_ar = info.get("name_ar", pretty_disease_name_en)

            if lang == "ar":
                if confidence > 85:
                    confidence_msg = "🔍 اكتشاف بدرجة ثقة عالية"
                elif confidence > 60:
                    confidence_msg = "ℹ️ اكتشاف بدرجة ثقة متوسطة"
                else:
                    confidence_msg = "⚠️ النتيجة منخفضة الدقة. يرجى تجربة صورة أخرى للحصول على دقة أفضل"
            else:
                if confidence > 85:
                    confidence_msg = "🔍 High confidence detection."
                elif confidence > 60:
                    confidence_msg = "ℹ️ Moderate confidence."
                else:
                    confidence_msg = "⚠️ Low confidence. Please try another image for better accuracy."



            # Fetch the latest sensor data from Google Drive
            sensor_data = get_sensor_data_from_drive()


            if sensor_data is None:
                msg = "❌ لم يتمكن من جلب بيانات الحساس. يرجى المحاولة لاحقاً." if lang == "ar" else "❌ Could not fetch sensor data. Please try again later."
                bot.send_message(chat_id, msg)
                return

            # Get sensor feedback (Check sensor ranges)
            # Convert raw sensor keys into readable format for range checking
            sensor_for_feedback = {
                "Air Temperature (°C)": sensor_data.get("temperature"),
                "Air Humidity (%)": sensor_data.get("humidity"),
                "Light (LDR Analog)": sensor_data.get("ldrAnalog"),
                "Soil Temperature (°C)": sensor_data.get("rs485_temperature"),
                "Soil Humidity (%)": sensor_data.get("rs485_humidity"),
                "Soil EC (dS/m)": sensor_data.get("rs485_conductivity"),
                "Soil pH": sensor_data.get("rs485_ph")
            }

            # Now get feedback with the correctly formatted keys
            sensor_feedback = get_sensor_feedback(sensor_for_feedback, plant_type, lang)
            advice = generate_disease_sensor_advice(predicted_disease, sensor_for_feedback, plant_type, lang)
            disease = disease_info.get(predicted_disease, {})
            # Basic sensors
            temperature = sensor_data.get('temperature', 'N/A')
            humidity = sensor_data.get('humidity', 'N/A')

            ldr_a = sensor_data.get('ldrAnalog', 'N/A')
            ldr_d = sensor_data.get('ldrDigital', 'N/A')

            # RS485 sensors
            rs_temp = sensor_data.get('rs485_temperature', 'N/A')
            rs_hum = sensor_data.get('rs485_humidity', 'N/A')
            rs_cond = sensor_data.get('rs485_conductivity', 'N/A')
            rs_ph = sensor_data.get('rs485_ph', 'N/A')

            info = disease_info.get(predicted_disease, {})
            crop_info = get_crop_info(plant_type)
            growth_stage = crop_info.get("growth_period", "N/A")
            growth_stage_translated = translate_growth_stage(growth_stage, lang)

            if lang == "ar":
                description = disease.get("description_ar", "لا توجد معلومات")
                cause = disease.get("cause_ar", "N/A")
                treatment = disease.get("treatment_ar", "N/A")
                response = f"""
📅 *نتيجة التحليل:*

🌿 *المحصول:* *{plant_type_ar}*

🦠 *المرض:* *{pretty_disease_name_ar}*\n*{pretty_disease_name_en}*


📊 *الدقة:* {confidence:.2f}%

{confidence_msg}

🌱 *مرحلة النمو:* {growth_stage_translated}

📊 *بيانات الحساسات:*
🌡️ درجة حرارة الهواء: {temperature} °C
💧 رطوبة الهواء: {humidity}%

🔦 قيمة الضوء : {ldr_a}
📺 شدة الضوء: {"مشرق" if ldr_d == 0 else "مظلم"}

🌡️ درجة حرارة التربة: {rs_temp} °C
💧 رطوبة التربة: {rs_hum}%
⚡ التوصيلية الكهربائية (EC): {rs_cond} µS/cm
🧪 مستوى الحموضة (pH): {rs_ph}

💡 *التفاصيل:* {description}
🤔 *السبب:* {cause}
💪 *العلاج:* {treatment}

🌱 *نتيجة الحساسات:*

{sensor_feedback}


🧠 *نصيحة إضافية:*
{advice}
"""
            else:
                description = disease.get("description_en", "No info")
                cause = disease.get("cause_en", "N/A")
                treatment = disease.get("treatment_en", "N/A")
                response = f"""
📅 *Analysis Result:*

🌿 *Plant:* {plant_type_en}
🦠 *Disease:* *{pretty_disease_name_en}*
📊 *Confidence:* {confidence:.2f}%

{confidence_msg}

🌱 *Growth Stage:* {growth_stage}

📊 *Sensor Data:*
🌡️ Air Temp: {temperature} °C
💧 Air Humidity: {humidity}%
🔦 LDR : {ldr_a}
📺 Light: {"Bright" if ldr_d == 0 else "Dark"}

🌡️ Soil Temperature: {rs_temp} °C
💧 Soil Humidity: {rs_hum}%
⚡ Soil Conductivity (EC): {rs_cond} µS/cm
🧪 Soil pH Level: {rs_ph}


💡 *Details:* {description}
🤔 *Cause:* {cause}
💪 *Treatment:* {treatment}

🌱 *Sensor Feedback:*

{sensor_feedback}

🧠 *Additional Advice:*
{advice}
"""

            bot.send_message(chat_id, response, parse_mode="Markdown")
            keyboard = types.InlineKeyboardMarkup()

            if lang == "ar":
                keyboard.add(types.InlineKeyboardButton("🔁 تحليل صورة جديدة", callback_data="/photo"))
                keyboard.add(types.InlineKeyboardButton("💡 الحصول على نصيحة", callback_data="/tips"))
                keyboard.add(types.InlineKeyboardButton("🌐 تغيير اللغة", callback_data="/language"))
                keyboard.add(types.InlineKeyboardButton("🌱 عرض بيانات الحساس", callback_data="/sensors"))
                keyboard.add(types.InlineKeyboardButton("🔁 اختيار نبتة أخرى", callback_data="/ChoosePlant"))


            else:
                keyboard.add(types.InlineKeyboardButton("🔁 Analyze Another Image", callback_data="/photo"))
                keyboard.add(types.InlineKeyboardButton("💡 Get Tips", callback_data="/tips"))
                keyboard.add(types.InlineKeyboardButton("🌐 Change My Language", callback_data="/language"))
                keyboard.add(types.InlineKeyboardButton("🌱 View Sensor Data", callback_data="/sensors"))
                keyboard.add(types.InlineKeyboardButton("🔁 Choose Another Plant", callback_data="/ChoosePlant"))



            msg_txt = "🧭 What would you like to do next?" if lang == "en" else "🧭 ماذا تريد أن تفعل بعد ذلك؟"
            bot.send_message(chat_id, msg_txt, reply_markup=keyboard)

        else:
            msg = "✅ لا يوجد مرض تم اكتشافه. المحصول يبدو صحياً." if lang == "ar" else "✅ No disease detected. Plant appears healthy."
            bot.send_message(chat_id, msg, parse_mode="Markdown")


    except Exception as e:
        bot.send_message(chat_id, "❌ Error analyzing image. Please retry.")
        print(f"Error: {e}")

    #user_choices.pop(chat_id, None)




@bot.message_handler(commands=['tips'])
def handle_tips(message):
    chat_id = message.chat.id
    lang = user_languages.get(chat_id, 'en')

    if chat_id not in user_choices:
        msg = "❌ Please choose a plant first using /ChoosePlant." if lang == 'en' else "❌ الرجاء اختيار نوع المحصول أولاً باستخدام /ChoosePlant."
        bot.send_message(chat_id, msg)
        return

    plant_type = user_choices[chat_id].lower()
    plant_type_en, plant_type_ar = get_plant_type_translations(plant_type)


    if lang == "ar":

        tips_by_plant = {
            "lettuce": """🌿 *دليل زراعة الخس*

*1. 🧱 تحضير التربة*
   استخدم تربة طينية رملية جيدة التصريف أو تربة حديقة مفككة مدعّمة بالسماد العضوي أو سماد الدود.
   حافظ على درجة الحموضة بين 6.0 و7.0 لنمو مثالي.

*2. 🌱 بدء البذور والزراعة*
   انقع البذور لمدة 4–6 ساعات قبل زراعتها بعمق 0.6 سم (1/4 إنش).
   📏 تباعد الزراعة:
   • الخس الورقي: 10–15 سم
   • الروماني/الرأسي: 15–30 سم
   قم بتخفيف الشتلات الضعيفة وازرع الشتلات بنفس العمق عند النقل.

*3. ☀️ بيئة النمو*
   استخدم أوعية بعمق 15 سم على الأقل.
   وفر 6–8 ساعات من ضوء الشمس يوميًا.
   في المناطق الحارة، وفر ظلًا بعد الظهر لتقليل الإجهاد.

*4. 💧 الري*
   حافظ على رطوبة التربة بشكل متساوٍ دائمًا.
   اسقِ في الصباح الباكر.
   تجنب الري من الأعلى لتقليل تعفن الأوراق والأمراض.

*5. 🌾 التسميد*
   استخدم سمادًا متوازنًا (10-10-10) عند الزراعة وعندما يصل ارتفاع الشتلات إلى حوالي 10 سم.
   أضف سمادًا عضويًا كل 2–3 أسابيع.
   ⚠️ تجنب الإفراط في التسميد بالنيتروجين لتقليل خطر الأمراض.

*6. 🌡️ درجة الحرارة ومنع الإزهار المبكر (التركيز على الأوراق بدل الزهور)*
   درجة الحرارة المثالية: 16–18°C.
   إذا ارتفعت الحرارة عن 24°C، استخدم نشارة أو قماش تظليل لتبريد التربة ومنع الإزهار المبكر.

*7. 🐛 مكافحة الآفات والأمراض*
   • طهر الأدوات بانتظام.
   • أزل الأوراق الميتة والأعشاب الضارة.
   • تجنب الزراعة المكتظة لتحسين التهوية.
   • رش زيت النيم كل 7–10 أيام.
   • استخدم صابون مبيد للحشرات أو تراب الدياتومي.
   • استخدم أغطية الصفوف الطافية لصد الحشرات.
   • استخدم مبيدات الفطريات النحاسية أو الكبريتية بحذر.

*8. ✂️ الحصاد*
   • احصد الأوراق الخارجية عندما تصل إلى 10–15 سم.
   • للخس الرأسي، احصد عندما تصبح الرأس صلبة وناضجة.
   • احصد في الصباح الباكر لأفضل نضارة.
   • أعد الزراعة كل 2–3 أسابيع لضمان إنتاج مستمر.
   • قم بتدوير أماكن الزراعة للحفاظ على صحة التربة.
""",

        "cucumber": """🥒 *دليل زراعة الخيار*

*1. 🧱 تحضير التربة*
   • استخدم تربة مفككة جيدة التصريف مثل التربة الرملية لتجنب تعفن الجذور.
   • أضف سمادًا عضويًا مثل الكومبوست أو روث الحيوانات المتحلل.
   • حافظ على درجة الحموضة بين **6.0–7.0**.
   • أضف وجبة العظام أو روث الدجاج المتحلل لتعزيز الإزهار والإثمار.

*2. 🌱 بدء البذور والزراعة*
   • انقع البذور لمدة **6–8 ساعات** لتحسين الإنبات.
   • ازرع بعمق **2.5 سم** بعد زوال خطر الصقيع.
   • 📏 المسافات: **30–60 سم** بين النباتات.
   • احتفظ بأقوى شتلتين في كل حفرة أو وعاء.
   • في حال النقل، تعامل مع الشتلات بلطف لأن الخيار حساس للجذور.

*3. ☀️ بيئة النمو*
   • استخدم أوعية بعمق **30 سم أو أكثر**.
   • اختر موقعًا مشمسًا وغنيًا بالكومبوست.
   • وفر **6–8 ساعات من الشمس المباشرة يوميًا**.
   • استخدم دعامات أو أقفاص لتحسين التهوية ونظافة الثمار.
   • استخدم نشارة ومصدات للرياح للحفاظ على الرطوبة في المناطق الجافة.

*4. 💧 الري*
   • حافظ على رطوبة التربة بشكل مستمر — الخيار يحب الماء.
   • اسقِ بعمق **2–3 مرات أسبوعيًا**، أكثر في الحر.
   • 🚫 لا تسقِ الأوراق — اسقِ عند قاعدة النبات لتقليل الأمراض.
   • استخدم النشارة للحفاظ على الرطوبة.

*5. 🌾 التسميد*
   • استخدم سمادًا متوازنًا (10-10-10) عند الزراعة.
   • عند بدء الإزهار، استخدم سمادًا منخفض النيتروجين وغني بالبوتاسيوم (مثل 5-10-10).
   • أضف كومبوست أو سماد دود كل **2–3 أسابيع**.
   • ⚠️ لا تفرط في النيتروجين لتجنب زيادة الأوراق على حساب الثمار.

*6. 🌡️ درجة الحرارة والسيطرة على النمو*
   • درجة الحرارة المثالية: **16–32°C**.
   • يجب أن تكون درجة حرارة التربة **15°C** قبل الزراعة.
   • استخدم أغطية الصفوف لحماية النباتات الصغيرة.
   • التدعيم يقلل من الأمراض ويحسن شكل الثمار.

*7. 🐛 إدارة الآفات والأمراض*

   🔧 *النظافة مهمة*
   • طهر الأدوات بالكحول أو المبيض.
   • أزل الأعشاب والأوراق الميتة والكروم القديمة.

   💨 *تحسين التهوية*
   • درّب الكروم على الدعامات.
   • خفف الأوراق الكثيفة لتجنب العفن.

   🌿 *العلاجات العضوية*
   • **زيت النيم**: ضد الحشرات والعفن — كل **7–10 أيام**.
   • **صابون مبيد للحشرات**: رش صباحًا ضد الحشرات الطرية.
   • **تراب الدياتومي**: يطرد الخنافس والبزاقات.
   • **أغطية الصفوف**: لمنع دخول الخنافس مبكرًا.
   • **رذاذ بيكربونات الصوديوم**: يمنع العفن (1 ملعقة صغيرة + 1 لتر ماء).

   ⚠️ *مشاكل شائعة*
   • *العفن البودري*: بقع بيضاء — استخدم زيت النيم أو مبيد فطري.
   • *خنافس الخيار*: صفراء مخططة — التقطها يدويًا أو استخدم أغطية.
   • *ذبول بكتيري*: تزال النباتات المصابة.
   • *تعفن نهاية الزهرة*: حافظ على الري المنتظم وأضف الكالسيوم.

*8. ✂️ نصائح الحصاد*
   • احصد الثمار بطول **10–20 سم** حسب النوع.
   • احصد كل **1–2 يوم** لتحفيز الإنتاج.
   • استخدم مقص أو قاطعة — لا تقطع باليد.
   • الخيار المتأخر يصبح **مرًّا وبذريًا** — احصد في الوقت المناسب.
   • **أعد الزراعة كل 2–3 أسابيع** لحصاد مستمر.
   • قم بتدوير المحاصيل للحفاظ على صحة التربة.
""",

            "tomato": """🍅 *نصائح زراعة الطماطم*

*1. 🧱 تحضير التربة*
   • استخدم تربة طينية رملية جيدة التصريف.
   • أضف كومبوست أو سماد دود للحصول على العناصر الغذائية.
   • حافظ على درجة الحموضة بين 6.0 و7.0.
   • أضف قشور البيض المطحونة أو وجبة العظام للكالسيوم.

*2. 🌱 بدء البذور والزراعة*
   • انقع البذور لمدة 4–6 ساعات قبل الزراعة.
   • ابدأ الزراعة في الداخل قبل 6–8 أسابيع من آخر صقيع.
   • ازرع بعمق 0.6 سم.
   • المسافات: الطماطم الشجرية 45–60 سم، المتسلقة 60–90 سم.
   • عند النقل، ازرع بعمق حتى أول ورقة حقيقية.
   • قم بإزالة الشتلات الضعيفة.

*3. ☀️ بيئة النمو*
   • استخدم أوعية بعمق 30–45 سم مع صرف جيد.
   • وفر 6–10 ساعات من ضوء الشمس المباشر.
   • استخدم دعائم أو أقفاص.
   • أدخل الكومبوست للتربة بانتظام.

*4. 💧 الري*
   • اسقِ بعمق مرة أو مرتين أسبوعيًا.
   • اسقِ في الصباح وعند قاعدة النبات.
   • استخدم النشارة للحفاظ على الرطوبة.

*5. 🌾 التسميد*
   • استخدم سماد متوازن (10-10-10) عند الزراعة.
   • بعد بدء الإزهار، استخدم سماد 5-10-10.
   • أضف كومبوست كل 2–3 أسابيع.
   • ⚠️ تجنب الإفراط في النيتروجين.

*6. 🌡️ درجة الحرارة والتحكم بالنمو*
   • درجة الحرارة المثالية: 21–29°C.
   • لا تزرع إذا كانت الليالي تحت 13°C.
   • استخدم أقمشة ظل في الحر.
   • قم بتقليم الأوراق السفلية لتحسين التهوية.

*7. 🐛 إدارة الآفات والأمراض*
   🔧 *النظافة:*
   • طهر الأدوات، أزل الأوراق والثمار التالفة.
   💨 *التهوية:*
   • افسح بين النباتات وقم بالتقليم.
   🌿 *العلاجات العضوية:*
   • زيت النيم، صابون مبيد، تراب الدياتومي.
   • أغطية الصفوف، مبيدات النحاس أو الكبريت عند الحاجة.
   ⚠️ *مشاكل شائعة:*
   • *اللفحة المبكرة*: أزل الأوراق المصابة.
   • *تعفن نهاية الزهرة*: أضف الكالسيوم وانتظم في الري.
   • *دودة الطماطم*: التقطها يدويًا أو استخدم Bt.

*8. ✂️ الحصاد*
   • احصد عندما يكون اللون مكتملًا والملمس طريًا.
   • لف أو اقطع لتجنب تلف الساق.
   • احصد يوميًا في موسم الذروة.
   • احصد قبل المطر لتفادي تشقق الثمار.
   • ازرع دفعات جديدة كل 4–6 أسابيع.
   • قم بتدوير المحاصيل لتحسين صحة التربة.

🌿 زراعة موفقة!
"""
        }
        general_tips = "📘 لمزيد من التعليمات، تأكد من فحص ظروف الحساسات واستخدام تحليل الصور."
    else:
        tips_by_plant = {
            "lettuce": """🌿 *Lettuce Growing Guide*

*1. 🧱 Soil Preparation*
   Use well-draining sandy loam or loose garden soil enriched with compost or worm castings.
   Maintain pH between 6.0–7.0 for optimal growth.

*2. 🌱 Seed Starting & Planting*
   Soak seeds for 4–6 hours before sowing 1/4 inch (0.6 cm) deep.
   📏 Spacing:
   • Leaf lettuce: 4–6 in (10–15 cm)
   • Romaine/Butterhead: 6–12 in (15–30 cm)
   Thin weak seedlings and transplant at the same depth when moving from indoors.

*3. ☀️ Growing Environment*
   Use containers at least 6 inches deep.
   Provide **6–8 hours of sunlight daily**. In hot climates, offer **afternoon shade** to prevent stress.

*4. 💧 Watering*
   Keep soil **evenly moist** at all times.
   Water in the **early morning**.
   Avoid overhead watering to prevent **leaf rot** and disease.

*5. 🌾 Fertilization*
   Use a **balanced 10-10-10 fertilizer** at planting and when seedlings reach ~10 cm.
   Side-dress with compost every **2–3 weeks**.
   ⚠️ Avoid over-fertilizing with nitrogen to reduce disease risk.

*6. 🌡️ Temperature & Bolting Prevention*
   Ideal temperature: **60–65°F (16–18°C)**.
   If temperatures rise above 75°F (24°C), apply mulch or use **shade cloth** to cool the soil and avoid **bolting**.

*7. 🐛 Pest & Disease Control*
   • Sanitize tools regularly.
   • Remove dead leaves and surrounding weeds.
   • Avoid overcrowding to improve airflow.
   • Spray **neem oil** every 7–10 days.
   • Use **insecticidal soap** or **diatomaceous earth** for pest control.
   • Apply **floating row covers** to block insects.
   • Use **copper** or **sulfur-based fungicides** cautiously.

*8. ✂️ Harvesting*
   • Harvest outer leaves when 4–6 inches (10–15 cm) long.
   • For head lettuce, cut when firm and mature.
   • Harvest in the **early morning** for best freshness.
   • **Resow every 2–3 weeks** for continuous supply.
   • Rotate planting beds to maintain **soil health**.
""",
            "cucumber": """🥒 *Cucumber Growing Guide*

*1. 🧱 Soil Preparation*
   • Use loose, well-draining soil like **sandy loam** to prevent root rot.
   • Enrich with organic matter such as **compost or aged manure**.
   • Maintain pH between **6.0–7.0** for optimal nutrient uptake.
   • Mix in **bone meal** or **composted chicken manure** to support flowering and fruiting.

*2. 🌱 Seed Starting & Planting*
   • Soak seeds in water for **6–8 hours** to enhance germination.
   • Sow **1 inch (2.5 cm)** deep after the last frost.
   • 📏 Space plants **30–60 cm (12–24 in)** apart in rows or mounds.
   • Thin seedlings to the **strongest 1–2 per hill/pot**.
   • If transplanting, handle gently — cucumbers dislike root disturbance.

*3. ☀️ Growing Environment*
   • Use containers **12+ inches deep** with proper drainage.
   • Choose a **sunny, compost-rich** garden location.
   • Provide **6–8 hours of direct sunlight** daily.
   • Use **trellises or cages** to save space, improve airflow, and keep fruits clean.
   • In windy/dry areas, apply **mulch** and use **windbreaks** to retain moisture.

*4. 💧 Watering*
   • Keep soil **consistently moist** — cucumbers are water-loving.
   • Water **deeply 2–3 times per week**, more during hot weather.
   • 🚫 Avoid watering leaves — always water at the **base** to reduce disease risk.
   • Mulch to **retain moisture** and suppress weeds.

*5. 🌾 Fertilization*
   • Apply **balanced fertilizer (10-10-10)** at planting.
   • Switch to **low-nitrogen, high-potassium** (e.g., 5-10-10) at flowering stage.
   • Add **compost or worm castings** every **2–3 weeks**.
   • ⚠️ Avoid excess nitrogen — it promotes leaves over fruits.

*6. 🌡️ Temperature & Growth Control*
   • Ideal growing temp: **16–32°C (60–90°F)**.
   • Soil temp must reach **15°C (59°F)** before planting.
   • Use **row covers** to protect young plants from cool nights or early pests.
   • **Trellising** helps reduce disease and improve fruit shape.

*7. 🐛 Pest & Disease Management*

   🔧 *Keep It Clean*
   • Sanitize tools with alcohol or bleach.
   • Remove **weeds, dead leaves, and old vines** regularly.

   💨 *Improve Airflow*
   • Train vines on trellises.
   • Thin dense foliage to prevent **mildew**.

   🌿 *Organic Treatments*
   • **Neem Oil**: For aphids, mites, mildew — apply every **7–10 days**.
   • **Insecticidal Soap**: Spray early morning on soft-bodied insects.
   • **Diatomaceous Earth**: Sprinkle to deter beetles, slugs, worms.
   • **Floating Row Covers**: Use early in the season to block beetles.
   • **Baking Soda Spray** (1 tsp + 1L water): Helps prevent powdery mildew.

   ⚠️ *Watch for Common Issues*
   • *Powdery mildew*: White leaf spots — treat with neem or fungicide.
   • *Cucumber beetles*: Yellow-striped — hand-pick or use covers.
   • *Bacterial wilt*: Spread by beetles — remove infected plants.
   • *Blossom end rot*: Avoid with **consistent watering** and **calcium-rich soil**.

*8. ✂️ Harvesting Tips*
   • Harvest when fruits are **10–20 cm (4–8 in)**, depending on variety.
   • Pick every **1–2 days** to encourage continuous production.
   • Use **scissors/snips** — avoid tearing fruit from the vine.
   • Overripe cucumbers become **bitter and seedy** — harvest on time.
   • For ongoing harvests, **replant every 2–3 weeks**.
   • Rotate crops each season to maintain **soil health** and avoid diseases.
""",
            "tomato": """🍅 *Tomato Growing Tips*

*🌱 1. Soil Preparation*
• Use sandy loam or loose garden soil to prevent waterlogging.
• Mix in compost or worm castings for nutrients.
• Maintain pH between 6.0 and 7.0.
• Add crushed eggshells or bone meal for calcium and to prevent blossom-end rot.

*🌱 2. Seed Starting & Planting*
• Soak seeds in water for 4–6 hours before sowing.
• Start seeds indoors 6–8 weeks before the last frost, 1/4 inch deep.
• Space plants: bush types 18–24", vine types 24–36".
• Transplant seedlings deep — up to first true leaves.
• Thin out weaker seedlings to promote strong growth.

*🌤 3. Growing Environment*
• Use containers 12–18" deep with drainage.
• Ensure 6–10 hours of direct sunlight daily.
• Use stakes, cages, or trellises to support plants.
• Enrich garden beds with compost.

*💧 4. Watering*
• Water deeply once or twice weekly; keep soil evenly moist.
• Water in the morning and at the base of the plant.
• Mulch around the plant to retain moisture and regulate temperature.

*🌿 5. Fertilization*
• Use a balanced fertilizer (10-10-10) at planting.
• Switch to 5-10-10 after flowering starts.
• Side-dress with compost every 2–3 weeks.
• Avoid excess nitrogen to encourage fruiting over leaf growth.

*🌡 6. Temperature & Growth Control*
• Ideal temperature: 21–29°C (70–84°F).
• Plant only when nights are above 13°C (55°F).
• Use shade cloth in hot climates (>32°C/90°F).
• Prune lower leaves and suckers on vine types for airflow and fruit boost.

*🐛 7. Pest & Disease Management*
🔧 *Keep It Clean:*
• Sterilize tools; remove dead leaves and fallen fruit.
💨 *Air Circulation:*
• Space plants, prune lower/inner leaves.
🌿 *Organic Tools:*
• Neem Oil (every 7–10 days): aphids, whiteflies, fungi.
• Insecticidal Soap: soft-bodied pests.
• Diatomaceous Earth: ground pests.
• Floating Row Covers: pest barrier.
• Copper/Sulfur Fungicide: for blight/mildew (only if needed).
⚠️ *Common Issues:*
• **Early blight:** Remove affected leaves, apply fungicide.
• **Blossom-end rot:** Add calcium, keep watering consistent.
• **Hornworms:** Hand-pick or use Bt spray.

*🍅 8. Harvesting Tips*
• Pick when fully colored and slightly soft.
• Twist or snip to avoid plant damage.
• Harvest daily in peak season.
• Pick before rain to avoid fruit splitting.
• For continuous harvests, plant new crops every 4–6 weeks.
• Rotate crops to protect soil health.

Happy growing! 🌿
"""

        }
        general_tips = "📘 For more info, check sensor conditions and use image analysis."

    tips = tips_by_plant.get(plant_type)


    #bot.send_message(chat_id, full_tips, parse_mode="HTML" if lang == "ar" else "Markdown")


    keyboard = types.InlineKeyboardMarkup(row_width=2)
    if lang == "ar":
        keyboard.add(types.InlineKeyboardButton("🔁  تحليل صورة  ", callback_data="/photo"))
        keyboard.add(types.InlineKeyboardButton("🌐 تغيير اللغة", callback_data="/language"))
        keyboard.add(types.InlineKeyboardButton("🌱 عرض بيانات الحساس", callback_data="/sensors"))
        keyboard.add(types.InlineKeyboardButton("🔁 اختيار نبتة أخرى", callback_data="/ChoosePlant"))

    else:
        keyboard.add(types.InlineKeyboardButton("🔁 Analyze Image", callback_data="/photo"))
        keyboard.add(types.InlineKeyboardButton("🌐 Change Language", callback_data="/language"))
        keyboard.add(types.InlineKeyboardButton("🌱 View Sensor Data", callback_data="/sensors"))
        keyboard.add(types.InlineKeyboardButton("🔁 Choose Another Plant", callback_data="/ChoosePlant"))

    bot.send_message(chat_id, tips, parse_mode="Markdown", reply_markup=keyboard)


# --- Start Bot ---
def start_polling():
    bot.polling(none_stop=True)

if __name__ == "__main__":
    print("🚀 Bot is running...")
    threading.Thread(target=start_polling).start()

✅ Google Drive authentication successful!
🚀 Bot is running...
